In [10]:
import numpy as np
import openmdao.api as om
import os

os.environ['OPENMDAO_REPORTS'] = 'none'

In [11]:
class turbineHeatTransfer(om.ExplicitComponent):
    def setup(self):
        # Global design variable
        # [Tc1, Tc2, Tc3, K, hle, hte, Plm, mdot, Tg, Fperf, Fecon]
        self.add_input('x', val=np.ones(11)) 

        # Coupling output
        self.add_output('Tbulk', val=1000.)

        # MATLAB FEM setup
        import turbineFEM
        import matlab
        self._double = matlab.double
        self._solveFEM = turbineFEM.initialize()
        self._geometry_filename = "turbine_blade.STL"

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        Tc1, Tc2, Tc3 = inputs['x'][0], inputs['x'][1], inputs['x'][2]
        K = inputs['x'][3]
        hle, hte = inputs['x'][4], inputs['x'][5]

        x_in = self._double(np.array([hle, hte, K, Tc1, Tc2, Tc3], dtype='d'), size=(1,6))

        outputs['Tbulk'] = self._solveFEM.turbineFEM(self._geometry_filename, x_in)

        with open("output.txt", "a") as text_file:
            text_file.write("heat %s\n" % outputs['Tbulk'])

class turbineLifetime(om.ExplicitComponent):
    def setup(self):
        # Global design variable
        # [Tc1, Tc2, Tc3, K, hle, hte, Plm, mdot, Tg, Fperf, Fecon]
        self.add_input('x', val=np.ones(11)) 

        # Coupling parameter
        self.add_input('Tbulk', val=1000.)

        # Coupling output
        self.add_output('tfail', val=1000.)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        Plm = inputs['x'][6]
        Tbulk = inputs['Tbulk']

        outputs['tfail'] = np.exp(Plm/Tbulk - 20)

class turbineLifetime_modified(om.ExplicitComponent):
    def setup(self):
        # Global design variable
        # [Tc1, Tc2, Tc3, K, hle, hte, Plm, mdot, Tg, Fperf, Fecon]
        self.add_input('x', val=np.ones(11)) 

        # Coupling parameter
        self.add_input('Tbulk', val=1000.)
        self.add_input('Peng', val=5.0e6)

        # Coupling output
        self.add_output('tfail', val=1000.)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        Plm = inputs['x'][6]
        Tbulk = inputs['Tbulk']
        Peng = inputs['Peng']

        outputs['tfail'] = np.exp(Plm/Tbulk - 20 + 2*(Peng/1e7)**2)

        with open("output.txt", "a") as text_file:
            text_file.write("life %s\n" % outputs['tfail'])

class turbinePerformance(om.ExplicitComponent):
    def setup(self):
        # Global design variable
        # [Tc1, Tc2, Tc3, K, hle, hte, Plm, mdot, Tg, Fperf, Fecon]
        self.add_input('x', val=np.ones(11)) 

        # Coupling output
        self.add_output('Peng', val=5.0e6)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        mdot, Tg, Fperf = inputs['x'][7], inputs['x'][8], inputs['x'][9]

        # Constants
        mdot0 = 30
        T0 = 300
        N = 90
        Cp = 1003.5

        outputs['Peng'] = Fperf*(mdot0-N*mdot)*Cp*T0*(1+Tg/T0-2*np.sqrt(Tg/T0))

class turbinePerformance_modified(om.ExplicitComponent):
    def setup(self):
        # Global design variable
        # [Tc1, Tc2, Tc3, K, hle, hte, Plm, mdot, Tg, Fperf, Fecon]
        self.add_input('x', val=np.ones(11)) 
        self.add_input('recon', val=1.0e4)
        self.add_input('tfail', val=100)

        # Coupling output
        self.add_output('Peng', val=5.0e6)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        mdot, Tg, Fperf = inputs['x'][7], inputs['x'][8], inputs['x'][9]
        recon = inputs['recon']
        tfail = inputs['tfail']

        # Constants
        mdot0 = 30
        T0 = 300
        N = 90
        Cp = 1003.5

        # outputs['Peng'] = Fperf*(mdot0-N*mdot)*Cp*T0*(1+Tg/T0-2*np.sqrt(Tg/T0))+ 0.0001*recon**2
        outputs['Peng'] = Fperf*(mdot0-N*mdot)*Cp*T0*(1+Tg/T0-2*np.sqrt(Tg/T0)) + 100*tfail**2 + 0.0001*recon**2

        with open("output.txt", "a") as text_file:
            text_file.write("perf %s\n" % outputs['Peng'])

class turbineEconomics(om.ExplicitComponent):
    def setup(self):
        # Global design variable
        # [Tc1, Tc2, Tc3, K, hle, hte, Plm, mdot, Tg, Fperf, Fecon]
        self.add_input('x', val=np.ones(11)) 

        # Coupling parameters
        self.add_input('tfail', val=1000.)
        self.add_input('Peng', val=5.0e6)

        # Coupling output
        self.add_output('recon', val=1000.)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        Fecon = inputs['x'][10]

        tfail = inputs['tfail']
        Peng = inputs['Peng']

        # Constant
        c0 = 0.07

        outputs['recon'] = Fecon*tfail*Peng*(c0/1000)

        with open("output.txt", "a") as text_file:
            text_file.write("econ %s\n" % outputs['recon'])

In [19]:
class turbineGroup(om.Group):
    def setup(self):
        self.add_subsystem('heat', turbineHeatTransfer(), promotes=['*'])
        self.add_subsystem('life', turbineLifetime(), promotes=['*'])
        self.add_subsystem('perf', turbinePerformance(), promotes=['*'])
        # self.add_subsystem('perf', turbinePerformance_modified(), promotes=['*'])
        self.add_subsystem('econ', turbineEconomics(), promotes=['*'])

        self.linear_solver = om.DirectSolver()
        self.nonlinear_solver = om.NonlinearBlockGS()

        self.linear_solver = om.DirectSolver()
        nlbgs = self.nonlinear_solver = om.NonlinearBlockGS()
        nlbgs.options['maxiter'] = 100

        open('output.txt', 'w').close()

class turbineGroup_feedback(om.Group):
    def setup(self):
        self.add_subsystem('heat', turbineHeatTransfer(), promotes=['*'])
        self.add_subsystem('life', turbineLifetime_modified(), promotes=['*'])
        self.add_subsystem('perf', turbinePerformance_modified(), promotes=['*'])
        self.add_subsystem('econ', turbineEconomics(), promotes=['*'])

        self.linear_solver = om.DirectSolver()
        nlbgs=self.nonlinear_solver = om.NonlinearBlockGS()
        nlbgs.options['maxiter'] = 100

        open('output.txt', 'w').close()

In [20]:
# model = om.Group()
# model.add_subsystem('heat', turbineHeatTransfer(), promotes=['*'])
# model.add_subsystem('life', turbineLifetime(), promotes=['*'])
# model.add_subsystem('perf', turbinePerformance(), promotes=['*'])
# model.add_subsystem('econ', turbineEconomics(), promotes=['*'])
prob = om.Problem(turbineGroup())
prob.setup()

# prob.set_val('x', np.array([600,650,700,30,2000,1000,2.5e4,0.12,1250,0.9,1.0]))
prob.set_val('x', np.array([590,640,690,29,1975,975,2.55e4,0.132,1225,0.9,1.0]))

prob.run_model()
print(prob['Tbulk'])
print(prob['tfail'])
print(prob['Peng'])
print(prob['recon'])


NL: NLBGS Converged in 2 iterations
[1099.29785712]
[24.44986903]
[5115141.35392063]
[8754.51753063]


In [21]:
# class turbineGroup(om.Group):
#     def setup(self):
#         self.add_subsystem('heat', turbineHeatTransfer(), promotes=['*'])
#         # self.add_subsystem('life', turbineLifetime(), promotes=['*'])
#         self.add_subsystem('life', turbineLifetime_modified(), promotes=['*'])
#         # self.add_subsystem('perf', turbinePerformance(), promotes=['*'])
#         self.add_subsystem('perf', turbinePerformance_modified(), promotes=['*'])
#         self.add_subsystem('econ', turbineEconomics(), promotes=['*'])

#         self.linear_solver = om.DirectSolver()
#         nlbgs = self.nonlinear_solver = om.NonlinearBlockGS()
#         nlbgs.options['maxiter'] = 100

#         open('output.txt', 'w').close()

In [22]:
# model = om.Group()
# model.add_subsystem('heat', turbineHeatTransfer(), promotes=['*'])
# model.add_subsystem('life', turbineLifetime(), promotes=['*'])
# model.add_subsystem('perf', turbinePerformance(), promotes=['*'])
# model.add_subsystem('econ', turbineEconomics(), promotes=['*'])
prob = om.Problem(turbineGroup())
prob.setup()

# prob.set_val('x', np.array([600,650,700,30,2000,1000,2.5e4,0.12,1250,0.9,1.0]))
prob.set_val('x', np.array([590,640,690,29,1975,975,2.55e4,0.132,1225,0.9,1.0]))

prob.run_model()
print(prob['Tbulk'])
print(prob['tfail'])
print(prob['Peng'])
print(prob['recon'])


NL: NLBGS Converged in 2 iterations
[1099.29785712]
[24.44986903]
[5115141.35392063]
[8754.51753063]


In [24]:
# Run problem, save residuals

import sys
import shutil
import torch
import numpy as np
from scipy.stats import qmc
from botorch.utils.transforms import unnormalize

prob = om.Problem()
prob.model = turbineGroup_feedback()

prob.driver = om.ScipyOptimizeDriver()
prob.driver.options['optimizer'] = 'COBYQA'
prob.driver.options['tol'] = 1e-8
prob.driver.options['disp'] = False

bounds = torch.tensor( [[590, 610], # Tc1
                        [640, 660], # Tc2
                        [690, 710], # Tc3
                        [29, 31], # K
                        [1975, 2025], # hle
                        [975, 1025], # hte
                        [2.45e4, 2.55e4], # Plm
                        [0.108, 0.132], # mdot
                        [1225, 1275], # Tg
                        [0.85, 0.95], # Fperf
                        [0.9, 1.1], # Fecon
                        [1000, 1200], # Tbulk
                        [0.01, 100], # tfail
                        [4.8e6, 6.6e6], # Peng
                        [0, 1.0e5]] # recon
                     ).transpose(0,1) 

prob.model.add_design_var('x', lower = bounds[0,:-4], upper = bounds[1,:-4])

prob.model.approx_totals()

# prob.setup()

# Generate input points
npts = 10
sampler = qmc.LatinHypercube(d=11, seed=1234)
x_input = unnormalize(torch.tensor(sampler.random(n=npts-1)), bounds=bounds[:,:-4])
x_input = torch.vstack((
    torch.tensor([590,640,690,29,1975,975,2.55e4,0.132,1225,0.9,1.0]),
    x_input))

for i, x in zip(range(0,npts), x_input):
    # prob.set_val('B', 233.1)
    prob.setup() # call this here to clear output.txt
    prob.set_val('x', x)

    prob.run_model()

    shutil.copy('output.txt', 'turbine_feedback_output' + str(i+1) + '.txt')
    print('evals:', prob.model.heat.iter_count + prob.model.heat.iter_count_apply +
         prob.model.life.iter_count + prob.model.life.iter_count_apply + 
         prob.model.perf.iter_count + prob.model.perf.iter_count_apply + 
         prob.model.econ.iter_count + prob.model.econ.iter_count_apply)

    with open('turbine_feedback_output' + str(i+1) + '.txt', 'r') as fp:
        line_count = sum(1 for line in fp)
    print('Total Number of lines:', line_count)
    
    # stdout = sys.stdout
    # with open('aerostrux_res_' + str(i+1) + '.txt', 'w') as sys.stdout:
    #     prob.run_model()
    # sys.stdout = stdout

# print('coupling vars')
# print('L', prob.get_val('L'))
# print('phi', prob.get_val('phi'))

# print('iter count', 
#       prob.model.cycle.aero.iter_count + prob.model.cycle.strux.iter_count)

NL: NLBGS Converged in 14 iterations
evals: 56
Total Number of lines: 56
NL: NLBGS Converged in 8 iterations
evals: 32
Total Number of lines: 32
NL: NLBGS Converged in 10 iterations
evals: 40
Total Number of lines: 40
NL: NLBGS Converged in 10 iterations
evals: 40
Total Number of lines: 40
NL: NLBGS Converged in 9 iterations
evals: 36
Total Number of lines: 36
NL: NLBGS Converged in 12 iterations
evals: 48
Total Number of lines: 48
NL: NLBGS Converged in 12 iterations
evals: 48
Total Number of lines: 48
NL: NLBGS Converged in 10 iterations
evals: 40
Total Number of lines: 40
NL: NLBGS Converged in 12 iterations
evals: 48
Total Number of lines: 48
NL: NLBGS Converged in 9 iterations
evals: 36
Total Number of lines: 36


In [ ]:
import numpy as np
bounds = np.array( [[590, 610], # Tc1
                    [640, 660], # Tc2
                    [690, 710], # Tc3
                    [29, 31], # K
                    [1975, 2025], # hle
                    [975, 1025], # hte
                    [2.45e4, 2.55e4], # Plm
                    [0.108, 0.132], # mdot
                    [1225, 1275], # Tg
                    [0.85, 0.95], # Fperf
                    [0.9, 1.1], # Fecon
                    [1000, 1200], # Tbulk
                    [0.01, 100], # tfail
                    [4.8e6, 6.6e6], # Peng
                    [0, 1.0e5]] ) # recon
bounds[:-4,0]